# Week 69, Day 2: Project: tokeniser

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w069d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('tiny_lex.py', 'w', encoding='utf-8') as f:
    f.write('from dataclasses import dataclass\n\n\nclass TinyError(Exception):\n    """Base class for every error the Tiny interpreter reports."""\n\n    def __init__(self, message, line=None, column=None):\n        super().__init__(message)\n        self.message, self.line, self.column = message, line, column\n\n    def __str__(self):\n        if self.line is None:\n            return self.message\n        if self.column is None:\n            return f"line {self.line}: {self.message}"\n        return f"line {self.line}, column {self.column}: {self.message}"\n\n\nclass LexError(TinyError):\n    """The source contains characters that don\'t form a token."""\n\n\nclass ParseError(TinyError):\n    """The tokens don\'t follow Tiny\'s grammar."""\n\n\nclass TinyRuntimeError(TinyError):\n    """Something went wrong while a program ran."""\n\n\n@dataclass(frozen=True)\nclass Token:\n    kind: str\n    text: str\n    value: object\n    line: int\n    column: int\n\n\nclass Cursor:\n    def __init__(self, source):\n        self.source, self.pos, self.line, self.column = source, 0, 1, 1\n\n    def at_end(self):\n        return self.pos >= len(self.source)\n\n    def peek(self, ahead=0):\n        i = self.pos + ahead\n        return self.source[i] if i < len(self.source) else ""\n\n    def advance(self):\n        ch = self.source[self.pos]\n        self.pos += 1\n        if ch == "\\n":\n            self.line, self.column = self.line + 1, 1\n        else:\n            self.column += 1\n        return ch\n\ndef read_number(cursor):\n    line, column, text = cursor.line, cursor.column, ""\n    while cursor.peek().isdigit():\n        text += cursor.advance()\n    if cursor.peek() == "." and cursor.peek(1).isdigit():\n        text += cursor.advance()\n        while cursor.peek().isdigit():\n            text += cursor.advance()\n        return Token("NUMBER", text, float(text), line, column)\n    return Token("NUMBER", text, int(text), line, column)\n\n\nESCAPES = {"n": "\\n", "t": "\\t", \'"\': \'"\', "\\\\": "\\\\"}\n\n\ndef read_string(cursor):\n    line, column = cursor.line, cursor.column\n    text = cursor.advance()\n    value = ""\n    while True:\n        if cursor.at_end() or cursor.peek() == "\\n":\n            raise LexError("unterminated string", line, column)\n        ch = cursor.advance()\n        text += ch\n        if ch == \'"\':\n            return Token("STRING", text, value, line, column)\n        if ch == "\\\\":\n            if cursor.at_end():\n                raise LexError("unterminated string", line, column)\n            esc_line, esc_col = cursor.line, cursor.column - 1\n            code = cursor.advance()\n            text += code\n            if code not in ESCAPES:\n                raise LexError(f"unknown escape \\\\{code}", esc_line, esc_col)\n            value += ESCAPES[code]\n        else:\n            value += ch\n')

### Exercise: Step 6: names and keywords

Write `read_name(cursor)`: with the cursor at a letter or underscore, read letters, digits and underscores. If the word is in `KEYWORDS`, return a token whose kind is the word in capitals (`LET`, `WHILE`, `TRUE`...) with value `True`, `False` or `None` for `true`, `false` and `nil` (and `None` for other keywords); otherwise a `NAME` token whose value is the name.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch

KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


def read_name(cursor):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_names():\n    \"names and keywords\"\n    c = Cursor(\"total_2 = 1\")\n    assert read_name(c) == Token(\"NAME\", \"total_2\", \"total_2\", 1, 1) and c.peek() == \" \"\n    assert read_name(Cursor(\"while(\")) == Token(\"WHILE\", \"while\", None, 1, 1)\n    assert read_name(Cursor(\"true\")).value is True and read_name(Cursor(\"false\")).value is False\n    assert read_name(Cursor(\"nil\")) == Token(\"NIL\", \"nil\", None, 1, 1)\n    assert read_name(Cursor(\"letter\")).kind == \"NAME\" and read_name(Cursor(\"_x\")).kind == \"NAME\"\n    assert read_name(Cursor(\"print\")).kind == \"PRINT\"")

### Exercise: Step 7: operators

Write `read_operator(cursor)`: read one of the two-character operators in `TWO_CHAR` (`==`, `!=`, `<=`, `>=`) if the next two characters make one, otherwise one of the single characters in `ONE_CHAR`, and return an `OP` token whose text and value are the operator. Anything else is a `LexError(f"unexpected character {ch!r}")` at that position. (`!` on its own isn't a Tiny operator: `not` is a keyword.)

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch

TWO_CHAR = {"==", "!=", "<=", ">="}
ONE_CHAR = set("+-*/%<>=(){},;")


def read_operator(cursor):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_operators():\n    \"one and two characters\"\n    c = Cursor(\"<=3\")\n    assert read_operator(c) == Token(\"OP\", \"<=\", \"<=\", 1, 1) and c.peek() == \"3\"\n    c = Cursor(\"= =\")\n    assert read_operator(c).text == \"=\" and c.peek() == \" \"\n    assert read_operator(Cursor(\"==\")).text == \"==\" and read_operator(Cursor(\"!=\")).text == \"!=\"\n    assert [read_operator(Cursor(ch)).text for ch in \"+-*/%<>(){},;\"] == list(\"+-*/%<>(){},;\")\n    for bad in (\"@\", \"!\", \"&\"):\n        try:\n            read_operator(Cursor(bad))\n            assert False\n        except LexError as e:\n            assert e.message == f\"unexpected character {bad!r}\" and (e.line, e.column) == (1, 1)")

### Exercise: Step 8: space and comments

Write `skip_space_and_comments(cursor)`: move the cursor past any spaces, tabs, carriage returns and newlines, and past comments (from `#` to the end of the line, not including the newline, which is then skipped as space), stopping at the first character that's part of a token, or at the end.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def skip_space_and_comments(cursor):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_skip():\n    \"skipping what isn't a token\"\n    c = Cursor(\"   \\t\\n  # a comment\\n\\n  let\")\n    skip_space_and_comments(c)\n    assert c.peek() == \"l\" and (c.line, c.column) == (4, 3)\n    c = Cursor(\"# only a comment\")\n    skip_space_and_comments(c)\n    assert c.at_end()\n    c = Cursor(\"x # trailing\")\n    skip_space_and_comments(c)\n    assert c.peek() == \"x\"\n    c = Cursor(\"\")\n    skip_space_and_comments(c)\n    assert c.at_end()")

### Exercise: Step 9: the tokeniser

Write `tokenize(source)`: a list of every token in the source, in order, using the functions from the earlier steps (given): skip space and comments; at the end, add an `EOF` token (text `""`, value `None`) at the cursor's position and stop; otherwise choose the reader by the next character: a digit starts a number, `"` a string, a letter or underscore a name, and anything else an operator.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch

def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch

KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tokenize():\n    \"the whole tokeniser\"\n    toks = tokenize('let x = 3.5;  # set x\\nprint \"hi\" + str(x) <= 4;')\n    assert [t.kind for t in toks] == [\"LET\", \"NAME\", \"OP\", \"NUMBER\", \"OP\", \"PRINT\", \"STRING\", \"OP\",\n                                      \"NAME\", \"OP\", \"NAME\", \"OP\", \"OP\", \"NUMBER\", \"OP\", \"EOF\"]\n    assert [t.text for t in toks[5:9]] == [\"print\", '\"hi\"', \"+\", \"str\"]\n    assert (toks[5].line, toks[5].column) == (2, 1) and toks[-1] == Token(\"EOF\", \"\", None, 2, 26)\n    assert tokenize(\"\") == [Token(\"EOF\", \"\", None, 1, 1)]\n    assert [t.kind for t in tokenize(\"if(not done)\")] == [\"IF\", \"OP\", \"NOT\", \"NAME\", \"OP\", \"EOF\"]\n    try:\n        tokenize(\"let y = 3 @ 4;\")\n        assert False\n    except LexError as e:\n        assert (e.line, e.column) == (1, 11)")

### Exercise: Step 10: every error at once

Stopping at the first error makes people fix their code one problem at a time. Write `tokenize_all(source)`, which returns `(tokens, errors)`: it works like `tokenize`, but when reading a token raises a `LexError`, it records the error and **recovers**: after an unexpected character, it skips that one character; after an unterminated string, it skips to the end of that line; after an unknown escape, it skips past the rest of the string (to the next unescaped `"` on the same line, or the end of the line). It always ends with an `EOF` token. Then `summary(errors)`: one line per error, `"line L, column C: message"`, joined with newlines.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch

def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch

KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return

def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))


def tokenize_all(source):
    ___

def summary(errors):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_recovery():\n    \"error recovery\"\n    toks, errors = tokenize_all('let a = 1 @ 2;\\nprint \"oops\\nlet b = 3 $;\\nprint \"x\\\\q\" + a;')\n    assert [str(e) for e in errors] == [\"line 1, column 11: unexpected character '@'\",\n                                        \"line 2, column 7: unterminated string\",\n                                        \"line 3, column 11: unexpected character '$'\",\n                                        r\"line 4, column 9: unknown escape \\q\"]\n    kinds = [t.kind for t in toks]\n    assert kinds[:6] == [\"LET\", \"NAME\", \"OP\", \"NUMBER\", \"NUMBER\", \"OP\"] and kinds[-1] == \"EOF\"\n    assert kinds[-5:] == [\"PRINT\", \"OP\", \"NAME\", \"OP\", \"EOF\"] and kinds.count(\"PRINT\") == 2\n    assert summary(errors).count(\"\\n\") == 3 and summary([]) == \"\"\n    good, none = tokenize_all(\"print 1;\")\n    assert none == [] and good == tokenize(\"print 1;\")")

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch

def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch

KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return

def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))


program = """fn square(n) {
    return n * n;   # times itself
}
print square(4) >= 16;
"""
for tok in tokenize(program):
    print(f"{tok.line}:{tok.column:<3} {tok.kind:7} {tok.text!r}")

> **One job per stage.** The tokeniser decides nothing about meaning. It doesn't know that `let` must be followed by a name, or that `(` needs a matching `)`. That's deliberate: each stage of the pipeline does one job, which makes each one small enough to test completely on its own. Tomorrow's parser can then assume its input is a clean list of tokens, each with a kind, a value and a position, and concentrate entirely on grammar.

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

So far you have the error family, `show_error`, `Token`, `Cursor`, `read_number` and `read_string`. They're collected here, and each exercise below gives you the code it needs:

## Step 6: names and keywords

Names (`total`, `make_counter`, `_x2`) and keywords (`let`, `while`, `true`) look the same to a reader: a letter or underscore followed by letters, digits and underscores. The difference is a lookup: if the word is a keyword, it gets its own token kind, which makes the parser's job much simpler (it can ask "is this a `WHILE`?" instead of comparing text). The values of `true`, `false` and `nil` are decided here too, once.

## Step 7: operators

Some operators are two characters long (`<=`, `==`), and some of those start with a character that's also an operator on its own (`<`, `=`). The tokeniser must look ahead and take the **longest** match, a rule called **maximal munch**: `<=` is one token, never `<` followed by `=`.

## Step 8: space and comments

Whitespace and comments separate tokens but aren't tokens themselves. The cursor still has to pass through them, though, so that line and column numbers stay right.

## Step 9: the tokeniser

Now the loop that ties it together. The next character decides what kind of token starts here, and each reader leaves the cursor just after its token, ready for the next one. The `EOF` token at the end saves the parser from checking "have I run out?" everywhere: it can always look at the next token.

## Step 10: every error at once

## Checkpoint

The finished tokeniser on a whole program, printed as a table:

## Quiz

## Before you go

Run your tokeniser in your head on `if(x<=-1)print"neg";` (no spaces): which tokens come out, and does any of them depend on maximal munch? Tomorrow: the parser, which turns these tokens into a syntax tree.